# HEXA UDON: train với target masking
Upload `HEXA_UDON_masked_20260915.zip` vào `/content`, bật GPU rồi chạy từng cell.
Gói chứa code hiện tại và checkpoint cuối seed43/44. Chúng có bộ đếm1068 nhưng chỉ học928 game từ weights mới.
Đây là bản để train kiểm chứng, chưa phải model được xác nhận mạnh hơn Greedy. Không chạy notebook cũ trong ZIP khác.


In [ ]:
from pathlib import Path
import os, zipfile
archive = Path('/content/HEXA_UDON_masked_20260915.zip')
with zipfile.ZipFile(archive) as z:
    z.extractall('/content/hexa_udon_masked_20260915')
repo = Path('/content/hexa_udon_masked_20260915/Procon2026')
os.chdir(repo)
print(repo)


In [ ]:
import sys, torch, numpy, requests
assert torch.cuda.is_available(), 'Hãy bật GPU runtime trước khi train'
print('Python:', sys.version)
print('Torch:', torch.__version__)
print('GPU:', torch.cuda.get_device_name(0))


Không tự cài hoặc nâng cấp package. Nếu import thất bại, kiểm tra `requirements.txt` và môi trường trước.
Checkpoint được chuyển sang GPU hiện tại khi load; không kỳ vọng tái lập từng bit giữa RTX4050 và GPU Colab.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
import subprocess
SEED = 44  # Đổi thành43 để chạy nhánh đối chứng còn lại.
INPUT = repo / f'runs/masked_pilot/seed{SEED}/additional_256.pt'
OUTPUT = Path(f'/content/drive/MyDrive/test_procon2026/masked_colab_20260915_seed{SEED}')
command = [sys.executable, '-u', 'report/pilot_training.py',
    '--model', str(INPUT), '--init-seed', str(SEED), '--episodes', '256', '--interval', '64',
    '--eval-seed', '13000', '--holdout-seed', '14000', '--eval-games', '100',
    '--output-dir', str(OUTPUT / 'reports'), '--checkpoint-dir', str(OUTPUT / 'checkpoints')]
print(' '.join(command))


Cell dưới train thêm256 game, lưu mỗi64 game vào Drive và đánh giá100 map mới.
Bộ lọc tự khôi phục từ checkpoint; batch16 game, gradient trung bình64 ngày/step.
Script từ chối dùng thư mục đầu ra đã tồn tại để tránh ghi đè kết quả. Khi chạy tiếp, đổi INPUT sang checkpoint mới nhất đã lưu và đổi OUTPUT sang một thư mục mới.


In [ ]:
subprocess.run(command, check=True)


In [ ]:
import json
report = json.loads((OUTPUT / 'reports/pilot.json').read_text())
for row in report['milestones']:
    print(row['additional_games'], {mode: (score['series'], score['wins'])
          for mode, score in row['evaluation'].items()})
print('Tập đối chiếu cuối:', {mode: (score['series'], score['wins'])
      for mode, score in report.get('final_holdout', {}).items()})


Xem thêm `report/MASKED_PILOT.md`. So với checkpoint đầu và Greedy; đừng chỉ dựa vào reward train hoặc một seed tốt.
Nếu Colab ngắt kết nối, chỉ checkpoint đã ghi xong trên Drive mới dùng để resume. Giữ các mốc trước khi đánh giá model mới.
